# GFF Parsing with Biopython (BCBio)

This notebook demonstrates how to parse a GFF3 file using `bcbio-gff` as suggested in the [Biopython Wiki](https://biopython.org/wiki/GFF_Parsing).

In [10]:
from BCBio import GFF
from Bio import SeqIO
import os
import pandas as pd
import os
import glob
import subprocess
import shutil
import gc

In [6]:

# Set the path to the GFF file
gff_file = "../viral_data_all/ncbi_dataset/data_subset/GCF_000864765.1/genomic.gff"

if not os.path.exists(gff_file):
    print(f"Error: {gff_file} not found.")
else:
    print(f"Found GFF file: {gff_file}")

Found GFF file: ../viral_data_all/ncbi_dataset/data_subset/GCF_000864765.1/genomic.gff


In [11]:
import os
import glob
import subprocess
import shutil

# 1. Define Paths
data_dir = '../viral_data_all/ncbi_dataset/data_subset/'

# 2. Identify the target FASTA files
# finding all files starting with GCA, GCF, ending with _genomic.fna, without 'cds'
file_pattern = '**/genomic.gff'

gff_files = [f for f in glob.glob(os.path.join(data_dir, file_pattern), recursive=True) if 'cds' not in os.path.basename(f)]

print(f"Found {len(gff_files)} GFF files matching the pattern.")

Found 211307 GFF files matching the pattern.


In [4]:
type(gff_files)

list

## Recursive Search for "slip"

GFF files often have nested features (e.g., CDS inside an mRNA inside a gene). To find all instances of "slip", we need to search recursively through all levels of features.

In [12]:
def search_features_recursive(features, search_term, record_id):
    results = []
    for feature in features:
        match = False
        
        # Check feature type
        if search_term in str(feature.type).lower():
            match = True
        
        # Check all qualifiers (attributes like 'Note', 'exception', etc.)
        if not match:
            for key, values in feature.qualifiers.items():
                for val in values:
                    if search_term in str(val).lower():
                        match = True
                        break
                if match: break
        
        if match:
            results.append({
                "record_id": record_id,
                "type": feature.type,
                "start": int(feature.location.start),
                "end": int(feature.location.end),
                "strand": feature.location.strand,
                "qualifiers": feature.qualifiers
            })
        
        # Recursively search through sub-features (children)
        if hasattr(feature, 'sub_features') and feature.sub_features:
            results.extend(search_features_recursive(feature.sub_features, search_term, record_id))
            
    return results

In [ ]:
##### SINGLE FILE TESTING ######

# all_slip_matches = []
# search_term = "slip"

# with open(gff_file) as in_handle:
#     for record in GFF.parse(in_handle):
#         matches = search_features_recursive(record.features, search_term, record.id)
#         all_slip_matches.extend(matches)

# print(f"Found {len(all_slip_matches)} total features matching '{search_term}'.")

# if all_slip_matches:
#     df_slip = pd.DataFrame(all_slip_matches)
#     display(df_slip.drop(columns=['qualifiers']).head())
    
#     print("\nFull attributes of the first match:")
#     import pprint
#     pprint.pprint(all_slip_matches[0]['qualifiers'])

In [16]:

all_matches = []
error_files = []
search_terms = ["slip"] # don't want 'ribosome_binding_site'
checkpoint_dir = "../gff_results/gff_checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)
checkpoint_interval = 10000
checkpoint_count = 0

for i, gff_file in enumerate(gff_files):
    try:
        with open(gff_file) as in_handle:
            for record in GFF.parse(in_handle):
                for search_term in search_terms:
                    matches = search_features_recursive(record.features, search_term, record.id)
                    for match in matches:
                        match["search_term"] = search_term
                        match["source_file"] = gff_file
                    all_matches.extend(matches)
    except Exception as e:
        print(f"Error parsing {gff_file}: {e}")
        error_files.append(gff_file)
        continue
    # Checkpoint: save and flush every 10,000 files
    if (i + 1) % checkpoint_interval == 0:
        if all_matches:
            df_chunk = pd.DataFrame(all_matches)
            df_chunk.to_csv(f"{checkpoint_dir}/checkpoint_{checkpoint_count}.csv", index=False)
            print(f"[{i+1}/{len(gff_files)}] Saved checkpoint_{checkpoint_count}.csv ({len(df_chunk)} rows)")
            checkpoint_count += 1
            all_matches = []  # free memory
            del df_chunk
            gc.collect()
        else:
            print(f"[{i+1}/{len(gff_files)}] No matches in this batch, skipping checkpoint.")
# Save any remaining matches after the loop
if all_matches:
    df_chunk = pd.DataFrame(all_matches)
    df_chunk.to_csv(f"{checkpoint_dir}/checkpoint_{checkpoint_count}.csv", index=False)
    print(f"[{len(gff_files)}/{len(gff_files)}] Saved final checkpoint_{checkpoint_count}.csv ({len(df_chunk)} rows)")
    del df_chunk
    gc.collect()
# --- Merge all checkpoints into one DataFrame ---
checkpoint_files = sorted(
    [f for f in os.listdir(checkpoint_dir) if f.startswith("checkpoint_") and f.endswith(".csv")]
)
print(f"\nMerging {len(checkpoint_files)} checkpoint files...")
df_all = pd.concat([pd.read_csv(f"{checkpoint_dir}/{f}") for f in checkpoint_files], ignore_index=True)
df_all = df_all.drop_duplicates(subset=["record_id", "type", "start", "end", "strand"])
print(f"Found {len(df_all)} unique features matching {search_terms} across {len(gff_files)} files.")
if error_files:
    print(f"Failed to parse {len(error_files)} files.")
display(df_all.drop(columns=['qualifiers']).head(20))

Error parsing ../viral_data_all/ncbi_dataset/data_subset/GCA_003108865.1/genomic.gff: Did not find remapped ID location: gene-tat, [[5846, 6061], [8371, 8462]], [5846, 8462]
[10000/211307] Saved checkpoint_0.csv (11797 rows)
Error parsing ../viral_data_all/ncbi_dataset/data_subset/GCA_003108965.1/genomic.gff: Did not find remapped ID location: gene-tat, [[5895, 6107], [8461, 8600]], [5895, 8600]
[20000/211307] Saved checkpoint_1.csv (11726 rows)
Error parsing ../viral_data_all/ncbi_dataset/data_subset/GCA_031239455.1/genomic.gff: Did not find remapped ID location: gene-tRNA-Arg (TCT), [[101529, 101568], [101482, 101519]], [101482, 101568]
[30000/211307] Saved checkpoint_2.csv (11801 rows)
[40000/211307] Saved checkpoint_3.csv (11785 rows)
[50000/211307] Saved checkpoint_4.csv (11814 rows)
[60000/211307] Saved checkpoint_5.csv (11802 rows)
Error parsing ../viral_data_all/ncbi_dataset/data_subset/GCF_000844405.1/genomic.gff: Did not find remapped ID location: gene-CcBV_6.3, [[14873, 1545

,record_id,type,start,end,strand,search_term,source_file
0,MG982796.1,CDS,24,594,1,slip,../viral_data_all/ncbi_dataset/data_subset/GCA...
1,MG982796.1,CDS,595,724,1,slip,../viral_data_all/ncbi_dataset/data_subset/GCA...
2,MT090511.1,CDS,0,570,1,slip,../viral_data_all/ncbi_dataset/data_subset/GCA...
3,MT090511.1,CDS,571,760,1,slip,../viral_data_all/ncbi_dataset/data_subset/GCA...
4,MK624178.1,CDS,12,582,1,slip,../viral_data_all/ncbi_dataset/data_subset/GCA...
5,MK624178.1,CDS,583,772,1,slip,../viral_data_all/ncbi_dataset/data_subset/GCA...
6,MN638426.1,CDS,12,582,1,slip,../viral_data_all/ncbi_dataset/data_subset/GCA...
7,MN638426.1,CDS,583,772,1,slip,../viral_data_all/ncbi_dataset/data_subset/GCA...
8,MW334173.1,CDS,24,594,1,slip,../viral_data_all/ncbi_dataset/data_subset/GCA...
9,MW334173.1,CDS,595,784,1,slip,../viral_data_all/ncbi_dataset/data_subset/GCA...


In [ ]:
### Checking what the errors are ################################################

import re
from collections import defaultdict
# Files that errored (extracted from your output)
print(error_files)
# --- Diagnose each file ---
for gff_file in error_files:
    print(f"\n{'='*80}")
    print(f"FILE: {gff_file}")
    print(f"{'='*80}")
    
    # Collect all feature lines and look for multi-part / trans-spliced genes
    feature_ids = defaultdict(list)
    
    try:
        with open(gff_file) as f:
            for line_num, line in enumerate(f, 1):
                if line.startswith("#"):
                    continue
                parts = line.strip().split("\t")
                if len(parts) < 9:
                    continue
                
                seqid, source, ftype, start, end, score, strand, phase, attributes = parts
                
                # Parse the ID and Parent from attributes
                attr_dict = {}
                for attr in attributes.split(";"):
                    if "=" in attr:
                        key, val = attr.split("=", 1)
                        attr_dict[key] = val
                
                feat_id = attr_dict.get("ID", "")
                parent = attr_dict.get("Parent", "")
                
                feature_ids[feat_id].append({
                    "line": line_num,
                    "type": ftype,
                    "start": start,
                    "end": end,
                    "strand": strand,
                    "parent": parent,
                })
        
        # Find features with the same ID appearing multiple times (discontinuous)
        print("\nFeatures with DUPLICATE IDs (multi-part/discontinuous locations):")
        found_any = False
        for feat_id, entries in feature_ids.items():
            if len(entries) > 1:
                found_any = True
                print(f"\n  ID: {feat_id} — appears {len(entries)} times")
                for e in entries:
                    print(f"    Line {e['line']}: {e['type']}  {e['start']}-{e['end']}  "
                          f"strand={e['strand']}  parent={e['parent']}")
        
        if not found_any:
            # Also check for join() in attributes or non-standard location encodings
            print("  (No duplicate IDs found — checking for 'join' or complex parents...)")
            
    except FileNotFoundError:
        print(f"  FILE NOT FOUND — skipping")
    except Exception as e:
        print(f"  Error reading file: {e}")

In [ ]:
import re
from collections import defaultdict

def fallback_parse_gff(gff_file, search_terms):
    """Raw text-based GFF parser for files that BCBio can't handle."""
    results = []
    with open(gff_file) as f:
        for line in f:
            if line.startswith("#"):
                continue
            parts = line.strip().split("\t")
            if len(parts) < 9:
                continue
            
            seqid, source, ftype, start, end, score, strand, phase, attributes = parts
            
            # Check if any search term matches the feature type or attributes
            line_lower = (ftype + "\t" + attributes).lower()
            for term in search_terms:
                if term in line_lower:
                    # Parse attributes into a dict
                    quals = {}
                    for attr in attributes.split(";"):
                        if "=" in attr:
                            k, v = attr.split("=", 1)
                            quals[k] = [v]
                    
                    results.append({
                        "record_id": seqid,
                        "type": ftype,
                        "start": int(start) - 1,  # GFF is 1-based, convert to 0-based
                        "end": int(end),
                        "strand": 1 if strand == "+" else (-1 if strand == "-" else 0),
                        "qualifiers": quals,
                        "search_term": term,
                        "source_file": gff_file,
                    })
                    break  # avoid duplicate matches for same line
    return results

# Deduplicate the file list
error_files = list(dict.fromkeys(error_files))

search_terms = ["slip", "ribosome", "ribosomal"]
fallback_matches = []

for gff_file in error_files:
    try:
        matches = fallback_parse_gff(gff_file, search_terms)
        fallback_matches.extend(matches)
        print(f"✓ {gff_file}: {len(matches)} matches found")
    except FileNotFoundError:
        print(f"✗ {gff_file}: FILE NOT FOUND")
    except Exception as e:
        print(f"✗ {gff_file}: {e}")

print(f"\nTotal fallback matches: {len(fallback_matches)}")

# Convert to DataFrame
if fallback_matches:
    df_fallback = pd.DataFrame(fallback_matches)
    df_fallback = df_fallback.drop_duplicates(subset=["record_id", "type", "start", "end", "strand"])
    print(f"Unique fallback features: {len(df_fallback)}")
    display(df_fallback.drop(columns=["qualifiers"]).head(20))

df_fallback.to_csv("../gff_results/gff_checkpoints/checkpoint_errors.csv", index=False)
print("Saved features to checkpoint_errors.csv")



✓ ../viral_data_all/ncbi_dataset/data_subset/GCA_003108865.1/genomic.gff: 0 matches found
✓ ../viral_data_all/ncbi_dataset/data_subset/GCA_003108965.1/genomic.gff: 2 matches found
✓ ../viral_data_all/ncbi_dataset/data_subset/GCA_031239455.1/genomic.gff: 0 matches found
✓ ../viral_data_all/ncbi_dataset/data_subset/GCF_000844405.1/genomic.gff: 0 matches found
✓ ../viral_data_all/ncbi_dataset/data_subset/GCA_002402265.1/genomic.gff: 0 matches found
✓ ../viral_data_all/ncbi_dataset/data_subset/GCF_000850265.1/genomic.gff: 0 matches found
✓ ../viral_data_all/ncbi_dataset/data_subset/GCA_003071505.1/genomic.gff: 0 matches found
✓ ../viral_data_all/ncbi_dataset/data_subset/GCA_000853945.1/genomic.gff: 0 matches found
✓ ../viral_data_all/ncbi_dataset/data_subset/GCA_000873645.1/genomic.gff: 0 matches found
✓ ../viral_data_all/ncbi_dataset/data_subset/GCF_003047555.1/genomic.gff: 0 matches found
✓ ../viral_data_all/ncbi_dataset/data_subset/GCA_000844405.1/genomic.gff: 0 matches found
✓ ../viral

,record_id,type,start,end,strand,search_term,source_file
0,AJ302647.1,CDS,843,2151,1,slip,../viral_data_all/ncbi_dataset/data_subset/GCA...
1,AJ302647.1,CDS,2150,5180,1,slip,../viral_data_all/ncbi_dataset/data_subset/GCA...
2,AJ302646.1,CDS,843,2151,1,slip,../viral_data_all/ncbi_dataset/data_subset/GCA...
3,AJ302646.1,CDS,2150,5180,1,slip,../viral_data_all/ncbi_dataset/data_subset/GCA...


Saved features to checkpoint_errors.csv


In [18]:
all_matches = []
search_terms = ["slip"] # don't want 'ribosome_binding_site'
checkpoint_dir = "../gff_results/gff_checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)
checkpoint_interval = 10000
checkpoint_count = 0

# --- Merge all checkpoints into one DataFrame ---
checkpoint_files = sorted(
    [f for f in os.listdir(checkpoint_dir) if f.startswith("checkpoint_") and f.endswith(".csv")]
)
print(f"\nMerging {len(checkpoint_files)} checkpoint files...")
df_all = pd.concat([pd.read_csv(f"{checkpoint_dir}/{f}") for f in checkpoint_files], ignore_index=True)
df_all = df_all.drop_duplicates(subset=["record_id", "type", "start", "end", "strand"])
print(f"Found {len(df_all)} unique features matching {search_terms} across {len(gff_files)} files and {len(checkpoint_files)} checkpoint files.")
## CHECK # OF CHECKPOINT FILES ####################
display(df_all.drop(columns=['qualifiers']).head(20))

df_all.to_csv("../gff_results/gff_parsed.csv")
print("Saved combined df to gff_parsed.csv")


Merging 23 checkpoint files...
Found 251146 unique features matching ['slip'] across 211307 files and 23 checkpoint files.


,record_id,type,start,end,strand,search_term,source_file
0,MG982796.1,CDS,24,594,1,slip,../viral_data_all/ncbi_dataset/data_subset/GCA...
1,MG982796.1,CDS,595,724,1,slip,../viral_data_all/ncbi_dataset/data_subset/GCA...
2,MT090511.1,CDS,0,570,1,slip,../viral_data_all/ncbi_dataset/data_subset/GCA...
3,MT090511.1,CDS,571,760,1,slip,../viral_data_all/ncbi_dataset/data_subset/GCA...
4,MK624178.1,CDS,12,582,1,slip,../viral_data_all/ncbi_dataset/data_subset/GCA...
5,MK624178.1,CDS,583,772,1,slip,../viral_data_all/ncbi_dataset/data_subset/GCA...
6,MN638426.1,CDS,12,582,1,slip,../viral_data_all/ncbi_dataset/data_subset/GCA...
7,MN638426.1,CDS,583,772,1,slip,../viral_data_all/ncbi_dataset/data_subset/GCA...
8,MW334173.1,CDS,24,594,1,slip,../viral_data_all/ncbi_dataset/data_subset/GCA...
9,MW334173.1,CDS,595,784,1,slip,../viral_data_all/ncbi_dataset/data_subset/GCA...


Saved combined df to gff_parsed.csv


# ANALYSIS

In [3]:
df = pd.read_csv("../gff_results/gff_parsed.csv")

In [9]:
df[df['search_term'] == 'slip']

,Unnamed: 0,record_id,type,start,end,strand,qualifiers,search_term,source_file
0,0,MG982796.1,CDS,24,594,1,"{'ID': ['cds-AVT55874.1'], 'Parent': ['gene-PA...",slip,../viral_data_all/ncbi_dataset/data_subset/GCA...
1,1,MG982796.1,CDS,595,724,1,"{'ID': ['cds-AVT55874.1'], 'Parent': ['gene-PA...",slip,../viral_data_all/ncbi_dataset/data_subset/GCA...
2,4,MT090511.1,CDS,0,570,1,"{'ID': ['cds-QIC52522.1'], 'Parent': ['gene-PA...",slip,../viral_data_all/ncbi_dataset/data_subset/GCA...
3,5,MT090511.1,CDS,571,760,1,"{'ID': ['cds-QIC52522.1'], 'Parent': ['gene-PA...",slip,../viral_data_all/ncbi_dataset/data_subset/GCA...
4,8,MK624178.1,CDS,12,582,1,"{'ID': ['cds-QBL91539.1'], 'Parent': ['gene-PA...",slip,../viral_data_all/ncbi_dataset/data_subset/GCA...
...,...,...,...,...,...,...,...,...,...
310449,561748,PQ011362.1,CDS,571,760,1,"{'ID': ['cds-XCO67773.1'], 'Parent': ['gene-PA...",slip,../viral_data_all/ncbi_dataset/data_subset/GCA...
310450,872201,AJ302647.1,CDS,843,2151,1,"{'ID': ['cds-CAC38421.2'], 'Parent': ['gene-ga...",slip,../viral_data_all/ncbi_dataset/data_subset/GCA...
310451,872202,AJ302647.1,CDS,2150,5180,1,"{'ID': ['cds-CAC38421.2'], 'Parent': ['gene-ga...",slip,../viral_data_all/ncbi_dataset/data_subset/GCA...
310452,872203,AJ302646.1,CDS,843,2151,1,"{'ID': ['cds-CAC38430.2'], 'Parent': ['gene-ga...",slip,../viral_data_all/ncbi_dataset/data_subset/GCA...
